In [0]:
# The S3 bucket path - replace with your actual S3 bucket name
#s3_source_path = "s3://your-bucket-name.s3.ap-southeast-2.amazonaws.com/orders/"
s3_source_path = "s3://databricksawsara/orders/"

# Use the existing raw_data_volume for checkpoints
checkpoint_path = "/Volumes/main/default/raw_data_volume/checkpoints/s3_ingest"

# Streaming from S3 using cloudFiles
df_s3_stream = (spark.readStream
  .format("cloudFiles")
  .option("cloudFiles.format", "json")
  .option("cloudFiles.schemaLocation", checkpoint_path)
  .load(s3_source_path)
)

# Writing to a Delta Table (table will be created automatically)
(df_s3_stream.writeStream
  .option("checkpointLocation", checkpoint_path)
  .trigger(availableNow=True)
  .toTable("main.default.bronze_sensors")
)

In [0]:
# Define the source and checkpoint paths using Unity Catalog Volumes
# Replace with your actual catalog, schema, and volume names
source_path = "/Volumes/main/default/raw_data_volume/exported_users_json/"
checkpoint_path = "/Volumes/main/default/raw_data_volume/checkpoints/user_data"

# Auto Loader Stream
df_stream = (spark.readStream
  .format("cloudFiles")
  .option("cloudFiles.format", "json") # The format of the source files
  .option("cloudFiles.schemaLocation", checkpoint_path) # Where to store schema info
  .load(source_path)
)

# Write the stream to a Delta Table
(df_stream.writeStream
  .option("checkpointLocation", checkpoint_path)
  .trigger(availableNow=True) # "availableNow" is the modern version of 'once=True'
  .toTable("main.default.silver_users")
)